# Point-vortex diagnostics
Plot circulation, Hamiltonian, geometry-relevant impulses, conservation drift, and cumulative dipole events.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

SCRIPT_DIRECTORY = Path.cwd() if (Path.cwd() / 'point_vortex_plotting.py').exists() else Path.cwd() / 'scripts'
sys.path.insert(0, str(SCRIPT_DIRECTORY.resolve()))
from point_vortex_plotting import (GEOMETRIES, filter_diagnostics, read_diagnostics,
    read_parameters, repository_root, rolling_mean, save_figure, use_plot_style)

## Configuration

In [ ]:
ROOT = repository_root()  # Repository root; normally no change is needed.
RUN_DIRECTORY = ROOT / 'runs/default'  # Completed solver run to analyze.
DIAGNOSTICS_FILE = RUN_DIRECTORY / 'diagnostics.csv'
PARAMETER_FILE = RUN_DIRECTORY / 'resolved_parameters.txt'
DIAGNOSTICS_FIGURE = RUN_DIRECTORY / 'figures/diagnostics.pdf'

# Geometry is normally inferred from PARAMETER_FILE. Set this explicitly to
# 'infinite', 'periodic', or 'disk' only to override the recorded geometry.
GEOMETRY = None
# Inclusive time interval, e.g. (0.1, 1.0). None keeps every time.
# Either bound may be None, e.g. (0.1, None).
TIME_RANGE = None
# Inclusive frame interval, e.g. (100, 1000). None keeps every frame.
# Either bound may be None.
FRAME_RANGE = None
# Number of saved samples in the centered moving average; 1 plots raw data.
ROLLING_WINDOW = 1
FIGURE_TITLE = None  # None creates a geometry-aware title.
USE_TEX = True       # False avoids requiring an external LaTeX installation.
FONT_SIZE = 14

## Load and validate the diagnostics

In [ ]:
if ROLLING_WINDOW < 1:
    raise ValueError('ROLLING_WINDOW must be positive')
for name, interval in (('TIME_RANGE', TIME_RANGE), ('FRAME_RANGE', FRAME_RANGE)):
    if interval is not None and len(interval) != 2:
        raise ValueError(f'{name} must contain exactly two bounds')
    if (interval is not None and interval[0] is not None and interval[1] is not None
            and interval[0] > interval[1]):
        raise ValueError(f'{name} requires its lower bound to be no greater than its upper bound')

use_plot_style(USE_TEX, FONT_SIZE)
parameters = read_parameters(PARAMETER_FILE) if PARAMETER_FILE.exists() else {}
geometry = GEOMETRY or parameters.get('boundaryCondition', 'infinite')
if geometry not in GEOMETRIES:
    raise ValueError(f"geometry must be one of {', '.join(GEOMETRIES)}")
diagnostics = filter_diagnostics(read_diagnostics(DIAGNOSTICS_FILE),
                                 time_range=TIME_RANGE, frame_range=FRAME_RANGE)
time = diagnostics['time']

def smooth(name):
    return rolling_mean(diagnostics[name], ROLLING_WINDOW)

if geometry == 'infinite':
    impulses = ['linear_impulse_x', 'linear_impulse_y', 'angular_impulse']
elif geometry == 'periodic':
    impulses = ['linear_impulse_x', 'linear_impulse_y']
else:
    impulses = ['angular_impulse']
print(f'Geometry: {geometry}; using {time.size} diagnostics samples')

## Invariants, conservation error, and dipole events

In [ ]:
value_labels = {
    'linear_impulse_x': r'$I_x$',
    'linear_impulse_y': r'$I_y$',
    'angular_impulse': r'$A$',
}
drift_symbols = {
    'linear_impulse_x': (r'I_x', r'I_{x,0}', r'I_{x,\mathrm{ref}}'),
    'linear_impulse_y': (r'I_y', r'I_{y,0}', r'I_{y,\mathrm{ref}}'),
    'angular_impulse': (r'A', r'A_0', r'A_{\mathrm{ref}}'),
}

def nonzero_absolute(values):
    absolute = np.abs(values)
    return np.where(absolute > 0.0, absolute, np.nan)

fig, axes_array = plt.subplots(2, 3, figsize=(14.0, 7.5), sharex=True)
axes = list(axes_array.flat)
axes[0].plot(time, smooth('circulation'), color='C0')
axes[0].set(ylabel=r'Circulation $\Gamma$', title='Total circulation')
axes[1].plot(time, smooth('hamiltonian'), color='black')
axes[1].set(ylabel=r'Hamiltonian $H$', title='Hamiltonian')
for name in impulses:
    axes[2].plot(time, smooth(name), label=value_labels[name])
axes[2].set(ylabel='Impulse', title='Geometry-relevant impulses')
axes[2].legend(frameon=False)

initial_errors = [('delta_circulation', r'$|\Gamma-\Gamma_0|$'),
                  ('delta_hamiltonian', r'$|H-H_0|$')]
segment_errors = [('segment_delta_circulation', r'$|\Gamma-\Gamma_{ref}|$'),
                  ('segment_delta_hamiltonian', r'$|H-H_{ref}|$')]
for name in impulses:
    symbol, initial_reference, segment_reference = drift_symbols[name]
    initial_errors.append((f'delta_{name}', rf'$|{symbol}-{initial_reference}|$'))
    segment_errors.append((f'segment_delta_{name}', rf'$|{symbol}-{segment_reference}|$'))
for name, label in initial_errors:
    axes[3].semilogy(time, nonzero_absolute(smooth(name)), label=label)
axes[3].set(ylabel='Absolute drift', title='Drift from initial state')
axes[3].legend(frameon=False)
for name, label in segment_errors:
    axes[4].semilogy(time, nonzero_absolute(smooth(name)), label=label)
axes[4].set(ylabel='Absolute drift', title='Drift from segment reference')
axes[4].legend(frameon=False)
axes[5].step(time, diagnostics['removed_pairs'], where='post', label='removed')
axes[5].step(time, diagnostics['reinjected_pairs'], where='post', label='re-injected')
axes[5].set(ylabel='Cumulative pairs', title='Dipole events')
axes[5].legend(frameon=False)
for axis in axes[3:]:
    axis.set_xlabel(r'Time $t$')
suffix = f' ({ROLLING_WINDOW}-sample rolling average)' if ROLLING_WINDOW > 1 else ''
fig.suptitle(FIGURE_TITLE or f'Point-vortex diagnostics: {geometry} geometry{suffix}')
saved = save_figure(fig, DIAGNOSTICS_FIGURE)
print(f'Wrote {saved}')
plt.show()